# 01. Exploratory Data Analysis (EDA)
## ICC Men's T20 World Cup 2022 Analytics

**Author:** Data Analyst Portfolio Project
**Dataset Source:** ESPNcricinfo Scraped Datasets

### Objective:
Conduct initial data profiling across raw JSON datasets covering 45 matches, batting innings, bowling spells, and player profiles.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

RAW_DATA_DIR = os.path.join('..', 'data', 'raw')
print(f'Raw data directory: {os.path.abspath(RAW_DATA_DIR)}')


### 1. Inspect Match Results (`t20_wc_match_results.json`)
Examines match metadata including teams, winners, victory margins, venues, and dates.


In [ ]:
match_path = os.path.join(RAW_DATA_DIR, 't20_wc_match_results.json')
with open(match_path, 'r', encoding='utf-8') as f:
    raw_matches = json.load(f)

df_matches = pd.DataFrame(raw_matches[0]['matchSummary'])
print('Matches shape:', df_matches.shape)
df_matches.head()


In [ ]:
print('Match summary info:')
print(df_matches.info())
print('\nUnique winners:\n', df_matches['winner'].value_counts())
print('\nUnique venues:\n', df_matches['ground'].value_counts())


### 2. Inspect Batting Summary (`t20_wc_batting_summary.json`)
Examines batsman-level innings logs across each fixture.


In [ ]:
bat_path = os.path.join(RAW_DATA_DIR, 't20_wc_batting_summary.json')
with open(bat_path, 'r', encoding='utf-8') as f:
    raw_bat = json.load(f)

bat_records = []
for m in raw_bat:
    bat_records.extend(m['battingSummary'])

df_bat = pd.DataFrame(bat_records)
print('Batting records shape:', df_bat.shape)
df_bat.head(10)


In [ ]:
print('Batting summary data types & null counts:')
print(df_bat.info())
print('\nSample batsman names with scraping encoding artifacts:')
print(df_bat[df_bat['batsmanName'].str.contains(r'[� †]|\(c\)', regex=True)]['batsmanName'].unique()[:10])


### 3. Inspect Bowling Summary (`t20_wc_bowling_summary.json`)
Examines bowler spells, overs bowled, wickets, and runs conceded.


In [ ]:
bowl_path = os.path.join(RAW_DATA_DIR, 't20_wc_bowling_summary.json')
with open(bowl_path, 'r', encoding='utf-8') as f:
    raw_bowl = json.load(f)

bowl_records = []
for m in raw_bowl:
    bowl_records.extend(m['bowlingSummary'])

df_bowl = pd.DataFrame(bowl_records)
print('Bowling records shape:', df_bowl.shape)
df_bowl.head(10)


In [ ]:
print('Bowling summary data types & info:')
print(df_bowl.info())


### 4. Inspect Player Master Dimension (`t20_wc_player_info.json`)
Examines player biographical details, roles, batting hand, and bowling style.


In [ ]:
player_path = os.path.join(RAW_DATA_DIR, 't20_wc_player_info.json')
with open(player_path, 'r', encoding='utf-8') as f:
    raw_players = json.load(f)

df_players = pd.DataFrame(raw_players)
print('Player profiles shape:', df_players.shape)
df_players.head(10)


In [ ]:
print('Playing roles distribution:')
print(df_players['playingRole'].value_counts())
print('\nBatting styles:\n', df_players['battingStyle'].value_counts())
print('\nBowling styles:\n', df_players['bowlingStyle'].value_counts())


### 5. Key Findings & Data Quality Issues Identified:
1. **Encoding artifacts in names**: Player names contain `\ufffd` (replacement character) and non-breaking spaces from scraping.
2. **Captain tags**: Captains have `(c)` appended to their names inconsistently across tables.
3. **String datatypes**: Numerical metrics (`runs`, `balls`, `wickets`, `overs`, `economy`) are stored as strings.
4. **Strike rate missing values**: In batting, strike rate is `-` when balls faced is 0.
5. **Duplicate player profiles**: 6 duplicate player records exist due to scraping both with and without captain tags.
6. **Match ID linking**: Batting and bowling tables reference fixtures as `'Team1 Vs Team2'` rather than scorecard IDs directly.
